# Phase 1: Data Preparation
**Speech Emotion Recognition — CREMA-D Dataset**

This notebook:
1. Scans all 7,442 WAV files in the CREMA-D dataset
2. Parses emotion labels from filenames
3. Creates a stratified train/val/test split (70/15/15)
4. Saves `metadata.csv` for use in all subsequent notebooks

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

print('Libraries loaded successfully.')

## 1. Configuration

In [ ]:
# Paths
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
CREMA_DIR = os.path.join(BASE_DIR, 'Crema')
OUTPUT_CSV = os.path.join(BASE_DIR, 'metadata.csv')

# Emotion mapping — CREMA-D label codes → integer class indices
EMOTION_MAP = {
    'ANG': 0,  # Anger
    'DIS': 1,  # Disgust
    'FEA': 2,  # Fear
    'HAP': 3,  # Happiness
    'NEU': 4,  # Neutral
    'SAD': 5   # Sadness
}

EMOTION_NAMES = {v: k for k, v in EMOTION_MAP.items()}
EMOTION_FULL  = {
    'ANG': 'Anger', 'DIS': 'Disgust', 'FEA': 'Fear',
    'HAP': 'Happiness', 'NEU': 'Neutral', 'SAD': 'Sadness'
}

print(f'CREMA-D directory: {CREMA_DIR}')
print(f'Emotion classes  : {list(EMOTION_MAP.keys())}')

## 2. Parse Filenames and Build Metadata DataFrame

File naming convention: `[ActorID]_[Sentence]_[Emotion]_[Intensity].wav`

Example: `1001_DFA_ANG_XX.wav` → Actor 1001, sentence DFA, Anger, intensity XX

In [ ]:
records = []

for fname in sorted(os.listdir(CREMA_DIR)):
    if not fname.endswith('.wav'):
        continue

    parts = fname.replace('.wav', '').split('_')
    if len(parts) != 4:
        continue  # skip malformed filenames

    actor_id, sentence, emotion_code, intensity = parts

    if emotion_code not in EMOTION_MAP:
        continue  # skip unknown emotion codes

    records.append({
        'filepath'     : os.path.join(CREMA_DIR, fname),
        'filename'     : fname,
        'actor_id'     : int(actor_id),
        'sentence'     : sentence,
        'emotion_code' : emotion_code,
        'emotion_label': EMOTION_MAP[emotion_code],
        'emotion_name' : EMOTION_FULL[emotion_code],
        'intensity'    : intensity,
    })

df = pd.DataFrame(records)
print(f'Total files parsed: {len(df)}')
df.head(10)

## 3. Dataset Overview

In [ ]:
print('=== Dataset Summary ===')
print(f'Total samples  : {len(df)}')
print(f'Unique actors  : {df["actor_id"].nunique()}')
print(f'Unique sentences: {df["sentence"].nunique()}')
print(f'Emotion classes: {df["emotion_code"].nunique()}')
print()
print('Class distribution:')
print(df.groupby(["emotion_code", "emotion_name"]).size().reset_index(name='count').to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — samples per emotion
emotion_counts = df['emotion_name'].value_counts()
axes[0].bar(emotion_counts.index, emotion_counts.values,
            color=['#e74c3c','#8e44ad','#2980b9','#27ae60','#95a5a6','#e67e22'])
axes[0].set_title('Samples per Emotion Class')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(axes[0].patches, emotion_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontsize=10)

# Pie chart
axes[1].pie(emotion_counts.values, labels=emotion_counts.index,
            autopct='%1.1f%%', startangle=140,
            colors=['#e74c3c','#8e44ad','#2980b9','#27ae60','#95a5a6','#e67e22'])
axes[1].set_title('Emotion Distribution (%)')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'emotion_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 4. Stratified Train / Val / Test Split (70 / 15 / 15)

We use stratified splitting to maintain class balance in every subset.

In [ ]:
# Step 1: train (70%) vs temp (30%)
df_train, df_temp = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['emotion_label']
)

# Step 2: val (15%) and test (15%) from the temp 30%
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, random_state=42, stratify=df_temp['emotion_label']
)

df_train = df_train.copy()
df_val   = df_val.copy()
df_test  = df_test.copy()

df_train['split'] = 'train'
df_val['split']   = 'val'
df_test['split']  = 'test'

print(f'Train : {len(df_train)} samples ({len(df_train)/len(df)*100:.1f}%)')
print(f'Val   : {len(df_val)} samples ({len(df_val)/len(df)*100:.1f}%)')
print(f'Test  : {len(df_test)} samples ({len(df_test)/len(df)*100:.1f}%)')

In [ ]:
# Verify class balance across splits
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, (subset_df, title) in zip(axes, [
    (df_train, 'Train'), (df_val, 'Validation'), (df_test, 'Test')
]):
    counts = subset_df['emotion_name'].value_counts()
    ax.bar(counts.index, counts.values,
           color=['#e74c3c','#8e44ad','#2980b9','#27ae60','#95a5a6','#e67e22'])
    ax.set_title(f'{title} Split')
    ax.set_xlabel('Emotion')
    ax.tick_params(axis='x', rotation=20)
    if ax == axes[0]:
        ax.set_ylabel('Count')

plt.suptitle('Class Distribution Across Splits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Class balance looks good if bars are approximately equal within each split.')

## 5. Save metadata.csv

In [ ]:
df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
df_all = df_all[['filepath', 'filename', 'actor_id', 'sentence',
                  'emotion_code', 'emotion_label', 'emotion_name', 'intensity', 'split']]

df_all.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')
print(f'Shape: {df_all.shape}')
df_all.head()

## 6. Sanity Check — Verify Files Exist

In [ ]:
missing = df_all[~df_all['filepath'].apply(os.path.exists)]
if len(missing) == 0:
    print(f'All {len(df_all)} audio files verified — no missing files.')
else:
    print(f'WARNING: {len(missing)} missing files!')
    print(missing[['filename', 'split']].head(10))

## Summary

| Item | Value |
|------|-------|
| Total samples | 7,442 |
| Emotion classes | 6 (ANG, DIS, FEA, HAP, NEU, SAD) |
| Train split | ~70% |
| Val split | ~15% |
| Test split | ~15% |
| Output | `metadata.csv` |

**Next step:** Run `feature_extraction.ipynb` to extract MFCC and Mel-Spectrogram features.